# Particle Trigger System Analysis
This notebook demonstrates the generation of synthetic particle collision data and evaluates three different trigger systems (Rule-based, Statistical, and Machine Learning) to separate physics signals from background noise.

In [ ]:
import os
import sys
# Add parent directory to path so we can import src
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.event_generator import EventGenerator
from src.trigger_rules import RuleBasedTrigger
from src.statistical_trigger import StatisticalTrigger
from src.ml_trigger import MLTrigger
from src.metrics import TriggerMetrics
from src.visualization import TriggerVisualizer

viz = TriggerVisualizer()

## 1. Event Generation
Let's generate 10,000 events with a 10% signal fraction.

In [ ]:
gen = EventGenerator(seed=123)
df = gen.generate_events(n_events=10000, signal_fraction=0.10, noise_scale=1.5)
print(df.head())

# Show correlation scatter
fig = viz.plot_feature_scatter(df)

## 2. Rule-Based Trigger Validation

In [ ]:
rule_trigger = RuleBasedTrigger(energy_threshold=40.0, max_noise_threshold=20.0)
preds_rule = rule_trigger.evaluate(df)

metrics_rule = TriggerMetrics.evaluate(df['signal_label'].values, preds_rule)
print("Rule-Based Metrics:", metrics_rule)

fig = viz.plot_confusion_matrix(df['signal_label'].values, preds_rule, title="Rule-Based CM")

## 3. Statistical Trigger Validation

In [ ]:
stat_trigger = StatisticalTrigger(p_value_threshold=0.01)
# Fit on mostly background
bkg_only = df[df['signal_label'] == 0]
stat_trigger.fit(bkg_only)

preds_stat = stat_trigger.evaluate(df)
metrics_stat = TriggerMetrics.evaluate(df['signal_label'].values, preds_stat)
print("Statistical Metrics:", metrics_stat)

fig = viz.plot_confusion_matrix(df['signal_label'].values, preds_stat, title="Statistical CM")

## 4. Machine Learning Trigger Validation

In [ ]:
# Using Random Forest
ml_trigger = MLTrigger(model_type='random_forest')

# Train/Test split
train_size = int(len(df) * 0.5)
df_train = df.iloc[:train_size]
df_test = df.iloc[train_size:]

ml_trigger.fit(df_train, df_train['signal_label'])
ml_trigger.set_thresholds(0.5)

preds_ml = ml_trigger.evaluate(df_test)
metrics_ml = TriggerMetrics.evaluate(df_test['signal_label'].values, preds_ml)
print("ML (Random Forest) Metrics:", metrics_ml)

fig = viz.plot_confusion_matrix(df_test['signal_label'].values, preds_ml, title="ML CM")